<a href="https://colab.research.google.com/github/zavisk/AutonomousMedicalImage_TriageAgent/blob/main/frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Medical Image Triage System
## Interactive Frontend — Powered by AI Ensemble + RAG + Agentic Workflow

## 1. Install Dependencies

In [ ]:
!pip install gradio chromadb sentence-transformers langgraph langchain langchain-core -q
print('All dependencies installed! ✅')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

## 2. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gradio as gr
from PIL import Image
from datetime import datetime
from typing import TypedDict, List, Dict
import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


## 3. Mount Drive & Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR     = '/content/drive/MyDrive/chexpert_outputs/'
TARGET_LABELS = ['Cardiomegaly', 'Pneumonia', 'Pneumothorax', 'Edema', 'Pleural Effusion']
IMG_SIZE      = 256
THRESHOLD     = 0.5

print(f'Model directory: {MODEL_DIR}')
print(f'Files: {os.listdir(MODEL_DIR)}')

Mounted at /content/drive
Model directory: /content/drive/MyDrive/chexpert_outputs/
Files: ['label_distribution.png', 'sample_images.png', 'densenet_121_best.pth', 'efficientnet_b0_best.pth', 'resnet_50_best.pth', 'efficientnet_b3_best.pth', 'per_class_all_models.png', 'training_curves.png', 'model_comparison.png', 'model_comparison.csv', 'config.json', 'training_history.json']


## 4. Model Classes

In [ ]:
class DenseNet121(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = models.densenet121(weights='IMAGENET1K_V1')
        in_f = self.model.classifier.in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_f, 512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x): return self.model(x)

class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = models.efficientnet_b0(weights='IMAGENET1K_V1')
        in_f = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_f, 512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x): return self.model(x)

class ResNet50(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = models.resnet50(weights='IMAGENET1K_V1')
        in_f = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_f, 512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x): return self.model(x)

class EfficientNetB3(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = models.efficientnet_b3(weights='IMAGENET1K_V1')
        in_f = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_f, 512), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x): return self.model(x)

MODELS = {
    'DenseNet-121':    DenseNet121,
    'EfficientNet-B0': EfficientNetB0,
    'ResNet-50':       ResNet50,
    'EfficientNet-B3': EfficientNetB3,
}
print('Model classes ready! ✅')

Model classes ready! ✅


## 5. Load Ensemble Models

In [ ]:
loaded_models = {}
for model_name, ModelClass in MODELS.items():
    m = ModelClass(num_classes=len(TARGET_LABELS))
    ckpt_path = os.path.join(MODEL_DIR, f"{model_name.lower().replace('-','_')}_best.pth")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        m.load_state_dict(ckpt['model_state_dict'])
        print(f'✅ {model_name} loaded')
    else:
        m.load_state_dict(ckpt)
        print(f'✅ {model_name} loaded')
    m.to(DEVICE).eval()
    loaded_models[model_name] = m

print(f'{len(loaded_models)} models loaded! ✅')

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 107MB/s]


✅ DenseNet-121 loaded
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 


✅ EfficientNet-B0 loaded
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]


✅ ResNet-50 loaded
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 149MB/s]


✅ EfficientNet-B3 loaded
4 models loaded! ✅


## 6. Transforms & Predictor

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def predict_ensemble(image_pil):
    image_tensor = val_transforms(image_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    all_probs = []
    with torch.no_grad():
        for model in loaded_models.values():
            probs = torch.sigmoid(model(image_tensor)).cpu().numpy()[0]
            all_probs.append(probs)
    avg_probs = np.mean(all_probs, axis=0)
    predictions = {label: float(prob) for label, prob in zip(TARGET_LABELS, avg_probs)}
    positive_findings = [l for l, p in predictions.items() if p >= THRESHOLD]
    return predictions, positive_findings

print('Predictor ready! ✅')

Predictor ready! ✅


## 7. RAG Pipeline

In [ ]:
class MedicalRAGPipeline:
    KNOWLEDGE_BASE = [
        {"id":"ptx_1","text":"Pneumothorax is the presence of air in the pleural space. Tension pneumothorax is life-threatening and requires immediate needle decompression. Treatment: chest tube insertion, oxygen therapy.","category":"Pneumothorax"},
        {"id":"ptx_2","text":"Pneumothorax urgency: large pneumothorax or tension pneumothorax is CRITICAL requiring emergency intervention within 15 minutes. Small pneumothorax may be monitored.","category":"Pneumothorax"},
        {"id":"edema_1","text":"Pulmonary edema is fluid accumulation in the lungs. Acute pulmonary edema is a medical emergency. Treatment: diuretics, oxygen, positioning, vasodilators.","category":"Edema"},
        {"id":"edema_2","text":"Pulmonary edema urgency: acute onset with respiratory distress is CRITICAL. Chronic stable edema is MODERATE. Requires immediate cardiology or ICU consultation.","category":"Edema"},
        {"id":"pna_1","text":"Pneumonia is an infection causing lung inflammation. Community-acquired pneumonia treated with antibiotics. Severe pneumonia with sepsis requires ICU admission.","category":"Pneumonia"},
        {"id":"pna_2","text":"Pneumonia urgency: severe pneumonia with hypoxia is URGENT requiring immediate antibiotics. Mild pneumonia is MODERATE with outpatient treatment.","category":"Pneumonia"},
        {"id":"cardio_1","text":"Cardiomegaly indicates underlying cardiac disease such as heart failure or cardiomyopathy. Requires echocardiogram and cardiology consultation.","category":"Cardiomegaly"},
        {"id":"cardio_2","text":"Cardiomegaly urgency: new onset with symptoms is URGENT. Chronic cardiomegaly is MODERATE requiring cardiology follow-up.","category":"Cardiomegaly"},
        {"id":"pe_1","text":"Pleural effusion is fluid in the pleural space. Causes include heart failure, infection, malignancy. Large effusions require thoracentesis.","category":"Pleural Effusion"},
        {"id":"pe_2","text":"Pleural effusion urgency: large effusion with respiratory distress is URGENT. Small effusion is MODERATE requiring outpatient investigation.","category":"Pleural Effusion"},
        {"id":"urgency_1","text":"CRITICAL: Immediate life-threatening condition requiring intervention within 15 minutes. Page on-call specialist immediately.","category":"Urgency"},
        {"id":"urgency_2","text":"URGENT: Serious condition requiring evaluation within 1 hour. MODERATE: Same-day evaluation. ROUTINE: Follow-up within 1 week.","category":"Urgency"}
    ]

    def __init__(self):
        print('Loading RAG pipeline...')
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection('medical_knowledge', metadata={'hnsw:space':'cosine'})
        if self.collection.count() == 0:
            texts = [d['text'] for d in self.KNOWLEDGE_BASE]
            embeddings = self.embedder.encode(texts).tolist()
            self.collection.add(
                ids=[d['id'] for d in self.KNOWLEDGE_BASE],
                embeddings=embeddings,
                documents=texts,
                metadatas=[{'category':d['category']} for d in self.KNOWLEDGE_BASE]
            )
        print(f'RAG ready with {self.collection.count()} documents! ✅')

    def get_evidence(self, positive_findings):
        if not positive_findings:
            return 'No significant findings. Patient appears normal based on AI analysis.'
        all_evidence = []
        for finding in positive_findings:
            query_emb = self.embedder.encode([f'{finding} clinical guidelines treatment urgency']).tolist()
            results = self.collection.query(query_embeddings=query_emb, n_results=2)
            all_evidence.extend(results['documents'][0])
        return '\n\n'.join(all_evidence[:4])

rag_pipeline = MedicalRAGPipeline()
print('RAG Pipeline ready! ✅')

Loading RAG pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG ready with 12 documents! ✅
RAG Pipeline ready! ✅


## 8. Triage Logic

In [ ]:
def classify_urgency(predictions):
    if predictions.get('Pneumothorax', 0) > 0.7:
        return 'CRITICAL', f"High confidence Pneumothorax ({predictions['Pneumothorax']:.1%}) - possible tension pneumothorax."
    elif predictions.get('Edema', 0) > 0.7:
        return 'CRITICAL', f"High confidence Pulmonary Edema ({predictions['Edema']:.1%}) - acute respiratory compromise."
    elif predictions.get('Pneumonia', 0) > 0.6:
        return 'URGENT', f"Pneumonia detected ({predictions['Pneumonia']:.1%}) - requires prompt antibiotic therapy."
    elif predictions.get('Pneumothorax', 0) > 0.5:
        return 'URGENT', f"Moderate confidence Pneumothorax ({predictions['Pneumothorax']:.1%}) - requires monitoring."
    elif predictions.get('Cardiomegaly', 0) > 0.5 or predictions.get('Pleural Effusion', 0) > 0.5:
        return 'MODERATE', 'Cardiomegaly or Pleural Effusion detected - requires specialist evaluation.'
    elif any(p > THRESHOLD for p in predictions.values()):
        return 'MODERATE', 'Findings detected - requires specialist evaluation.'
    return 'ROUTINE', 'No significant findings detected. Routine follow-up recommended.'

def determine_routing(predictions, urgency):
    if urgency == 'CRITICAL':
        return 'Emergency Medicine + ICU', 'Critical condition requires immediate emergency intervention.'
    elif predictions.get('Pneumothorax', 0) > 0.5:
        return 'Thoracic Surgery', 'Pneumothorax requires thoracic surgery evaluation.'
    elif predictions.get('Edema', 0) > 0.5:
        return 'Cardiology', 'Pulmonary edema likely cardiac in origin.'
    elif predictions.get('Pneumonia', 0) > 0.5:
        return 'Pulmonology / Infectious Disease', 'Pneumonia requires pulmonology management.'
    elif predictions.get('Cardiomegaly', 0) > 0.5:
        return 'Cardiology', 'Cardiomegaly requires cardiac workup and echocardiogram.'
    elif predictions.get('Pleural Effusion', 0) > 0.5:
        return 'Pulmonology', 'Pleural effusion requires investigation and possible thoracentesis.'
    return 'General Medicine', 'Routine evaluation recommended.'

print('Triage logic ready! ✅')

Triage logic ready! ✅


## 9. Chart Generators

In [ ]:
def create_probability_chart(predictions):
    labels = list(predictions.keys())
    probs  = [predictions[l] * 100 for l in labels]
    colors = ['#e74c3c' if p >= 50 else '#2ecc71' for p in probs]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(labels, probs, color=colors, edgecolor='white', linewidth=1.5)

    for bar, prob in zip(bars, probs):
        ax.text(min(prob + 1, 95), bar.get_y() + bar.get_height()/2,
                f'{prob:.1f}%', va='center', fontsize=11, fontweight='bold')

    ax.axvline(x=50, color='orange', linestyle='--', linewidth=2)
    ax.set_xlim(0, 105)
    ax.set_xlabel('Probability (%)', fontsize=12)
    ax.set_title('AI Ensemble Prediction - Disease Probability', fontsize=14, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3)

    positive_patch = mpatches.Patch(color='#e74c3c', label='POSITIVE')
    negative_patch = mpatches.Patch(color='#2ecc71', label='Negative')
    threshold_line = plt.Line2D([0],[0], color='orange', linestyle='--', label='Threshold 50%')
    ax.legend(handles=[positive_patch, negative_patch, threshold_line], fontsize=9, loc='lower right')

    plt.tight_layout()
    return fig

def create_model_agreement_chart(image_pil):
    image_tensor = val_transforms(image_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    individual_preds = {}
    with torch.no_grad():
        for name, model in loaded_models.items():
            probs = torch.sigmoid(model(image_tensor)).cpu().numpy()[0]
            individual_preds[name] = probs * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(TARGET_LABELS))
    width = 0.2
    colors = ['#3498db','#e67e22','#2ecc71','#e74c3c']

    for idx, (name, probs) in enumerate(individual_preds.items()):
        ax.bar(x + idx*width, probs, width, label=name, color=colors[idx], alpha=0.85)

    ax.axhline(y=50, color='black', linestyle='--', linewidth=1.5, label='Threshold (50%)')
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(TARGET_LABELS, rotation=15, ha='right', fontsize=10)
    ax.set_ylabel('Probability (%)', fontsize=11)
    ax.set_title('Individual Model Predictions - Ensemble Agreement', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    return fig

print('Chart generators ready! ✅')

Chart generators ready! ✅


## 10. Main Analysis Function

In [ ]:
URGENCY_COLORS = {
    'CRITICAL': '🔴',
    'URGENT':   '🟠',
    'MODERATE': '🟡',
    'ROUTINE':  '🟢',
}

def analyze_xray(image, patient_id):
    if image is None:
        return None, None, 'Please upload a chest X-ray image.', '', '', '', ''

    if not patient_id or not patient_id.strip():
        patient_id = f'PATIENT-{datetime.now().strftime("%H%M%S")}'

    image_pil = Image.fromarray(image) if not isinstance(image, Image.Image) else image

    predictions, positive_findings = predict_ensemble(image_pil)
    evidence = rag_pipeline.get_evidence(positive_findings)
    urgency, urgency_reason = classify_urgency(predictions)
    specialist, routing_reason = determine_routing(predictions, urgency)

    prob_chart = create_probability_chart(predictions)
    agreement_chart = create_model_agreement_chart(image_pil)

    findings_lines = []
    for label, prob in predictions.items():
        icon = '🔴 POSITIVE' if prob >= THRESHOLD else '🟢 Negative'
        bar  = 'X' * int(prob * 20) + '.' * (20 - int(prob * 20))
        findings_lines.append(f'{icon}  {label:<20} [{bar}]  {prob:.1%}')
    findings_text = '\n'.join(findings_lines)

    emoji = URGENCY_COLORS[urgency]
    urgency_text = (
        f'{emoji} URGENCY LEVEL: {urgency}\n\n'
        f'Reason: {urgency_reason}\n\n'
        f'Refer to: {specialist}\n'
        f'Why: {routing_reason}'
    )

    if urgency == 'CRITICAL':
        alert = f'CRITICAL ALERT - Page {specialist} immediately for {patient_id}! Intervention required within 15 MINUTES!'
    elif urgency == 'URGENT':
        alert = f'URGENT - Notify {specialist} for {patient_id}. Evaluation required within 1 HOUR.'
    elif urgency == 'MODERATE':
        alert = f'MODERATE - Schedule {specialist} consult for {patient_id}. Same-day evaluation recommended.'
    else:
        alert = f'ROUTINE - No urgent findings for {patient_id}. Schedule routine follow-up within 1 week.'

    evidence_text = f'RETRIEVED CLINICAL EVIDENCE (via RAG)\n{chr(45)*50}\n{evidence}'

    positive_str = ', '.join(positive_findings) if positive_findings else 'None'
    preds_lines = ''.join([f'{l}: {p:.1%}\n' for l, p in predictions.items()])
    report = (
        f'MEDICAL TRIAGE REPORT\n'
        f'Patient ID   : {patient_id}\n'
        f'Timestamp    : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        f'AI Model     : Ensemble (4 Models, AUC: 0.8825)\n\n'
        f'POSITIVE FINDINGS: {positive_str}\n\n'
        f'ALL PREDICTIONS:\n{preds_lines}\n'
        f'URGENCY      : {urgency} {emoji}\n'
        f'REASON       : {urgency_reason}\n\n'
        f'SPECIALIST   : {specialist}\n'
        f'ROUTING      : {routing_reason}\n\n'
        f'CLINICAL EVIDENCE (RAG):\n{evidence[:500]}...'
    )

    return prob_chart, agreement_chart, findings_text, urgency_text, alert, evidence_text, report

print('Main analysis function ready! ✅')

Main analysis function ready! ✅


## 11. Launch Gradio Frontend

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title='Medical Image Triage System') as demo:

    gr.Markdown("""
    # 🏥 Autonomous Medical Image Triage System
    ### Powered by AI Ensemble + RAG + Agentic Workflow
    **Upload a chest X-ray and get an instant AI-powered triage report**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 📤 Upload Chest X-Ray')
            image_input  = gr.Image(label='Chest X-Ray Image', type='pil', height=300)
            patient_id   = gr.Textbox(label='Patient ID (optional)', placeholder='e.g. PATIENT-001')
            analyze_btn  = gr.Button('🔍 Analyze X-Ray', variant='primary', size='lg')
            gr.Markdown('### ⚠️ Alert')
            alert_output = gr.Textbox(label='', lines=3, interactive=False)

        with gr.Column(scale=2):
            gr.Markdown('### 🔬 AI Findings')
            findings_output = gr.Textbox(label='Disease Detection Results', lines=7, interactive=False)
            gr.Markdown('### 🚦 Triage Decision')
            urgency_output  = gr.Textbox(label='Urgency and Routing', lines=6, interactive=False)

    gr.Markdown('---')
    gr.Markdown('## 📊 AI Analysis Charts')
    with gr.Row():
        prob_chart_output      = gr.Plot(label='Disease Probability Chart')
        agreement_chart_output = gr.Plot(label='Model Agreement Chart')

    gr.Markdown('---')
    gr.Markdown('## 📚 Clinical Evidence and Full Report')
    with gr.Row():
        with gr.Column():
            evidence_output = gr.Textbox(label='RAG Retrieved Clinical Evidence', lines=10, interactive=False)
        with gr.Column():
            report_output   = gr.Textbox(label='Full Triage Report', lines=10, interactive=False)

    gr.Markdown('---')
    with gr.Accordion('How Does This System Work?', open=False):
        gr.Markdown("""
        ## Pipeline Steps
        1. Upload chest X-ray image
        2. 4 deep learning models analyze independently (DenseNet-121, EfficientNet-B0, ResNet-50, EfficientNet-B3)
        3. Predictions are averaged (Ensemble AUC: 0.8825, above 0.85 baseline)
        4. RAG retrieves relevant clinical guidelines from knowledge base
        5. Rule-based AI assigns urgency: CRITICAL / URGENT / MODERATE / ROUTINE
        6. Automatically routes to correct specialist
        7. Structured report + alerts generated

        ## Conditions Detected
        - Cardiomegaly: Enlarged heart
        - Pneumonia: Lung infection
        - Pneumothorax: Air in pleural space (life threatening)
        - Edema: Fluid in lungs
        - Pleural Effusion: Fluid around lungs
        """)

    analyze_btn.click(
        fn=analyze_xray,
        inputs=[image_input, patient_id],
        outputs=[
            prob_chart_output,
            agreement_chart_output,
            findings_output,
            urgency_output,
            alert_output,
            evidence_output,
            report_output
        ]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_16499/3602141919.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='Medical Image Triage System') as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e2fbad3f65985edc88.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e2fbad3f65985edc88.gradio.live
